# pkgconfig smoke test

`pkgconfig` is a Python interface to the `pkg-config` command-line tool. Example 1 needs nothing else. Examples 2-4 write their own `.pc` files into a temporary directory and point `pkg-config` at that directory only, so they do not depend on any library installed on the machine. They need a `pkg-config` (or `pkgconf`) executable on `PATH`, and print a note instead of running when there is none.

## Example 1: the public API, and the error when `pkg-config` cannot be run

In [ ]:
# Example 1: the public API, and the error raised when the pkg-config executable cannot be run.
import importlib.metadata
import os

import pkgconfig

print(f"pkgconfig {importlib.metadata.version('pkgconfig')}")
API = (
    "exists", "modversion", "cflags", "libs", "parse", "installed", "requires", "variables",
    "list_all", "configure_extension",
)
missing = [name for name in API if not callable(getattr(pkgconfig, name, None))]
assert not missing, f"pkgconfig no longer exposes {missing}"

error = pkgconfig.PackageNotFoundError("libdemo")
assert isinstance(error, Exception) and str(error) == "libdemo not found", str(error)

# pkgconfig runs the executable named by $PKG_CONFIG (default `pkg-config`). When that cannot be run,
# a query raises OSError rather than answering "not installed".
saved_pkg_config = os.environ.get("PKG_CONFIG")
os.environ["PKG_CONFIG"] = os.path.join(os.getcwd(), "no-such-pkg-config")
try:
    pkgconfig.exists("zlib")
except OSError as exc:
    print(f"missing executable -> {type(exc).__name__}: {exc}")
else:
    raise AssertionError("exists() answered although the pkg-config executable does not exist")
finally:
    if saved_pkg_config is None:
        os.environ.pop("PKG_CONFIG", None)
    else:
        os.environ["PKG_CONFIG"] = saved_pkg_config
print("Example 1 OK")

## Example 2: `exists`, `modversion`, `cflags` and `libs`

In [ ]:
# Example 2: exists, modversion, cflags and libs against two .pc files written by this cell.
# PKG_CONFIG_LIBDIR replaces pkg-config's default search path, so only these files are visible and the
# answers do not depend on what is installed on the machine. Examples 3 and 4 reuse them.
import os
import shutil
import tempfile
from pathlib import Path

import pkgconfig

PKG_CONFIG_EXE = os.environ.get("PKG_CONFIG") or "pkg-config"
HAVE_PKG_CONFIG = shutil.which(PKG_CONFIG_EXE) is not None

PC_DIR = Path(tempfile.mkdtemp(prefix="pkgconfig-demo-"))
(PC_DIR / "demo.pc").write_text(
    "prefix=/opt/demo\n"
    "includedir=${prefix}/include\n"
    "libdir=${prefix}/lib\n"
    "\n"
    "Name: demo\n"
    "Description: A made-up library for this test\n"
    "Version: 1.2.3\n"
    "Cflags: -I${includedir} -DDEMO_API=1 -DDEMO_SHARED\n"
    "Libs: -L${libdir} -ldemo\n"
    "Libs.private: -lm\n"
)
(PC_DIR / "demo-plugin.pc").write_text(
    "Name: demo-plugin\n"
    "Description: A made-up plugin that needs demo\n"
    "Version: 0.4\n"
    "Requires: demo >= 1.0\n"
    "Libs: -ldemoplugin\n"
)
SAVED_ENV = {key: os.environ.get(key) for key in ("PKG_CONFIG_LIBDIR", "PKG_CONFIG_PATH")}
os.environ["PKG_CONFIG_LIBDIR"] = str(PC_DIR)
os.environ["PKG_CONFIG_PATH"] = ""  # no extra directories in front of PC_DIR

if not HAVE_PKG_CONFIG:
    print(f"`{PKG_CONFIG_EXE}` is not on PATH. pkgconfig runs it for every query, so Examples 2-4 only print this note.")
else:
    assert pkgconfig.exists("demo") is True
    assert pkgconfig.exists("this-package-does-not-exist") is False
    assert pkgconfig.modversion("demo") == "1.2.3", pkgconfig.modversion("demo")

    cflags = pkgconfig.cflags("demo")
    assert set(cflags.split()) == {"-I/opt/demo/include", "-DDEMO_API=1", "-DDEMO_SHARED"}, cflags
    libs = pkgconfig.libs("demo")
    assert libs.split() == ["-L/opt/demo/lib", "-ldemo"], libs
    # static=True adds Libs.private.
    assert "-lm" in pkgconfig.libs("demo", static=True).split()
    # demo-plugin pulls in demo's flags through its `Requires:` line.
    plugin_libs = pkgconfig.libs("demo-plugin")
    assert {"-ldemoplugin", "-ldemo"} <= set(plugin_libs.split()), plugin_libs

    try:
        pkgconfig.modversion("this-package-does-not-exist")
    except pkgconfig.PackageNotFoundError as exc:
        print(f"missing package -> {exc}")
    else:
        raise AssertionError("modversion() answered for a package that does not exist")
    print(f"cflags: {cflags}")
    print(f"libs:   {libs}")
    print("Example 2 OK")

## Example 3: `parse` and `configure_extension`

In [ ]:
# Example 3: parse() splits the flags into the fields a setuptools Extension takes, and
# configure_extension() appends them to an Extension. Uses the .pc files written in Example 2.
import types

import pkgconfig

if HAVE_PKG_CONFIG:
    info = pkgconfig.parse("demo")
    assert info["include_dirs"] == ["/opt/demo/include"], dict(info)
    assert info["library_dirs"] == ["/opt/demo/lib"], dict(info)
    assert info["libraries"] == ["demo"], dict(info)
    assert set(info["define_macros"]) == {("DEMO_API", "1"), ("DEMO_SHARED", None)}, dict(info)
    assert set(pkgconfig.parse("demo", static=True)["libraries"]) == {"demo", "m"}

    # Several packages at once, as a space-separated string.
    both = pkgconfig.parse("demo demo-plugin")
    assert set(both["libraries"]) == {"demo", "demoplugin"}, dict(both)

    # configure_extension() only touches `extra_compile_args` and `extra_link_args`, so any object with
    # those two lists stands in for setuptools.Extension here.
    ext = types.SimpleNamespace(extra_compile_args=[], extra_link_args=[])
    pkgconfig.configure_extension(ext, "demo")
    assert set(ext.extra_compile_args) == {"-I/opt/demo/include", "-DDEMO_API=1", "-DDEMO_SHARED"}, ext.extra_compile_args
    assert ext.extra_link_args == ["-L/opt/demo/lib", "-ldemo"], ext.extra_link_args
    print(f"parse: {dict(info)}")
    print("Example 3 OK")

## Example 4: `installed`, `requires`, `variables` and `list_all`

In [ ]:
# Example 4: version checks with installed(), the Requires: line, .pc variables and list_all().
# Uses the .pc files written in Example 2, then removes them and restores the environment.
import os
import shutil

import pkgconfig

try:
    if HAVE_PKG_CONFIG:
        assert pkgconfig.installed("demo", ">= 1.0") is True
        assert pkgconfig.installed("demo", "== 1.2.3") is True
        assert pkgconfig.installed("demo", "< 1.2") is False
        assert pkgconfig.installed("this-package-does-not-exist", ">= 0") is False

        assert pkgconfig.requires("demo-plugin") == ["demo >= 1.0"], pkgconfig.requires("demo-plugin")

        variables = pkgconfig.variables("demo")
        expected = {"prefix": "/opt/demo", "includedir": "/opt/demo/include", "libdir": "/opt/demo/lib"}
        assert expected.items() <= variables.items(), variables

        # PC_DIR is the only directory on the search path, so both files written in Example 2 are listed.
        listed = pkgconfig.list_all()
        assert {"demo", "demo-plugin"} <= set(listed), listed
        print(f"variables: {variables}")
        print(f"list_all:  {sorted(listed)}")
        print("Example 4 OK")
finally:
    for key, value in SAVED_ENV.items():
        if value is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = value
    shutil.rmtree(PC_DIR, ignore_errors=True)